# QuBridge - 再現性 notebook

論文の 4 つのデータ artifact を、`../data/` にキャッシュされた CSV から再現する。

| Section | 論文 artifact | CSV |
|---|---|---|
| 1 | Table I + Fig 2(b) - パイプライン trace | `table1_pipeline_trace.csv` |
| 2 | Table II + Fig 3(b) - L2 フィルタカスケード | `table2_l2_filter.csv` |
| 3 | Fig 4(b) - L3 ゲート別パルス波形 | `fig4b_l3_pulse_shape.csv` |
| 4 | Table III + Fig 5 - 符号化テレポーテーション | `table3_encoded_teleportation.csv` |

**IBM Quantum 実機アクセス不要。** すべての simulation 出力は CSV にキャッシュ済み。

`Cell -> Run All` で順次実行、または section 単位で実行する。

## セットアップ

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path('..') / 'data'
assert DATA.is_dir(), f'data/ not found at {DATA.resolve()}'
list(DATA.glob('*.csv'))

## 1. Table I + Fig 2(b) - パイプライン trace

3 段階の進行的アブレーション：

- Stage 0 (Baseline)：184 個の L2 path をサンプル、L3 は All Square 固定
- Stage 1 (+L2)：L2 をノイズ考慮 path (66, 67, 74) に固定、L3 を 4 種類の波形設定でスイープ
- Stage 2 (+L2+L3)：両者ともゲート別最適に固定

**Band** = F* − Min F。F* = 0.985 はジョイントパイプライン最適値で、全行に渡る共通の上限参照として使う。

In [ ]:
df1 = pd.read_csv(DATA / 'table1_pipeline_trace.csv')
f_star = 0.985  # joint L2+L3 pipeline optimum

g1 = df1.groupby(['stage', 'stage_label']).agg(
    n=('fidelity', lambda s: s.nunique()),
    min_F=('fidelity', 'min'),
    max_F=('fidelity', 'max'),
).reset_index()
g1['band_pp'] = (f_star - g1['min_F']) * 100
g1.round(3)

**期待値（論文 Table I）：**

| Layer | n | Min F | Max F | Band (pp) |
|---|---|---|---|---|
| Baseline | 184 | 0.867 | 0.982 | 11.8 |
| +L2 | 4 | 0.975 | 0.985 | 1.0 |
| +L2+L3 | 1 | 0.985 | 0.985 | 0.0 |

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
x = range(len(g1))
ax.fill_between(x, g1['min_F'], g1['max_F'], color='#94A3B8', alpha=0.3, label='Fidelity band')
ax.plot(x, g1['max_F'], 'o-', color='#16A34A', label='Max F')
ax.plot(x, g1['min_F'], 's-', color='#2563EB', label='Min F')
ax.axhline(f_star, color='#94A3B8', linestyle=':', label=f'Pipeline optimum F={f_star}')
ax.set_xticks(x)
ax.set_xticklabels(g1['stage_label'])
ax.set_ylabel('Fidelity F')
ax.set_title('Pipeline Convergence (Fig 2b)')
ax.legend(loc='lower right', fontsize=8)
ax.grid(axis='y', linestyle=':', alpha=0.5)
fig.tight_layout()
plt.show()

## 2. Table II + Fig 3(b) - L2 フィルタカスケード

IBM Torino 上での累積閾値フィルタリング。L3 は終始 All Square 固定。各フィルタレベルで残った最良/最悪 path を見る。

In [ ]:
df2 = pd.read_csv(DATA / 'table2_l2_filter.csv')
filters = df2['filter_label'].drop_duplicates().tolist()

g2 = df2.groupby('filter_label').agg(
    n_paths=('n_paths', 'first'),
    worst_F=('fidelity', 'min'),
    best_F=('fidelity', 'max'),
).reindex(filters)
g2['band_pp'] = (f_star - g2['worst_F']) * 100
g2.round(3)

**期待値（論文 Table II）：**

| Filter | Paths | Worst F | Best F | Band (pp) |
|---|---|---|---|---|
| Baseline | 184 | 0.867 | 0.976 | 11.8 |
| +RO<5% | 67 | 0.935 | 0.976 | 5.0 |
| 2Q<1% | 63 | 0.952 | 0.976 | 3.3 |
| 2Q<0.5% | 48 | 0.961 | 0.976 | 2.4 |
| 2Q<0.3% | 27 | 0.966 | 0.976 | **1.9** |

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
x = range(len(g2))
ax.fill_between(x, g2['worst_F'], g2['best_F'], color='#94A3B8', alpha=0.3,
                label='Fidelity band (L3=All Sq.)')
ax.plot(x, g2['best_F'], 'o-', color='#16A34A', label='Best F')
ax.plot(x, g2['worst_F'], 's-', color='#2563EB', label='Worst F')
ax.axhline(f_star, color='#94A3B8', linestyle=':',
           label=f'Pipeline optimum F={f_star}')
ax.set_xticks(x)
ax.set_xticklabels(g2.index, rotation=20, ha='right', fontsize=8)
ax.set_ylabel('Fidelity F')
ax.set_title('L2 Cumulative Filtering (Fig 3b)')
ax.legend(loc='lower right', fontsize=8)
ax.grid(axis='y', linestyle=':', alpha=0.5)
fig.tight_layout()
plt.show()

## 3. Fig 4(b) - L3 ゲート別パルス波形比較

3 種の一様 proxy 設定（All Square / All Gaussian Sq. / All DRAG）と、ゲート別最適 proxy（SX=DRAG, CZ=Gaussian Sq., Meas=Square）を比較する。

データソース：`table1_pipeline_trace.csv` の Stage 1（L2 をノイズ考慮 path に固定、L3 を 4 設定でスイープ）。

In [ ]:
# Stage 1 of e1h has L3 swept across 4 settings with L2 fixed.
df4 = df1[df1['stage'] == 1].copy()
df4.head()

In [ ]:
g4 = df4.groupby('pulse_config')['fidelity'].mean().sort_values()
g4_pct = g4 * 100
print(g4_pct.round(2).to_string())

**期待値（論文 Fig 4(b)）：** ゲート別最適は F=98.49% を達成（All Square ベースライン比 +0.87%）。

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
colors = ['#E24B4A' if 'Square' in c and 'Gaussian' not in c else
          '#3FA7E6' if 'Gaussian' in c else
          '#EF9F27' if 'DRAG' in c.upper() else '#1D9E75' for c in g4.index]
ax.bar(range(len(g4)), g4_pct, color=colors)
ax.set_xticks(range(len(g4)))
ax.set_xticklabels(g4.index, rotation=15, ha='right', fontsize=8)
ax.set_ylabel('Fidelity (%)')
ax.set_title('L3 Per-gate Pulse-Shape Proxy (Fig 4b)')
ax.set_ylim(g4_pct.min() - 0.5, g4_pct.max() + 0.5)
for i, v in enumerate(g4_pct):
    ax.text(i, v + 0.05, f'{v:.2f}', ha='center', fontsize=8)
fig.tight_layout()
plt.show()

## 4. Table III + Fig 5 - bit-flip / parity 検出テレポーテーション

物理（3 量子ビット）と [[2,1,1]] 符号化（6 量子ビット）テレポーテーションを、2 つの入力状態（|+>, |1>）で複数のノイズスケールにわたって比較する。

- |+> は位相敏感 — Z error は本符号で検出されない
- |1> は T1 減衰敏感 — bit-flip によるパリティ変化が syndrome filter で検出される

In [ ]:
df3 = pd.read_csv(DATA / 'table3_encoded_teleportation.csv')
df3.columns.tolist()

In [ ]:
# Pivot by (state_label, circuit_type); acceptance = 1 - error_detection_rate.
df3['acceptance'] = 1 - df3['error_detection_rate']
table3 = df3.pivot_table(
    index='noise_scale',
    columns=['state_label', 'circuit_type'],
    values=['fidelity', 'acceptance'],
    aggfunc='first',
)
table3.round(4)

**期待値（論文 Table III）：** noise=1.0 にて、|+> Phys.=0.9849, Log.=0.9769, Accept=0.9263；|1> Phys.=0.9844, Log.=0.9941, Accept=0.9244。

In [ ]:
# Fig 5: physical vs logical (conditional) fidelity per state vs noise.
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), sharey=True)
for ax, state, title in zip(axes, ['|+>', '|1>'],
                            ['|+> (Z-error dominant)', '|1> (T1 decay)']):
    sub = df3[df3['state_label'] == state].drop_duplicates(['noise_scale', 'circuit_type'])
    phys = sub[sub['circuit_type'] == 'physical'].sort_values('noise_scale')
    log = sub[sub['circuit_type'] == 'logical'].sort_values('noise_scale')
    ax.plot(phys['noise_scale'], phys['fidelity'], 'o-', label='Physical', color='#2563EB')
    ax.plot(log['noise_scale'], log['fidelity'], 's-', label='Logical (cond.)', color='#16A34A')
    ax.set_xlabel('Noise scale')
    ax.set_title(title)
    ax.grid(linestyle=':', alpha=0.5)
    ax.legend(fontsize=8)
axes[0].set_ylabel('Fidelity F')
fig.suptitle('Bit-Flip/Parity-Detected Teleportation (Fig 5)')
fig.tight_layout()
plt.show()

## 完了

論文の 4 つの artifact を、キャッシュされた simulation CSV からすべて再現した。データ ↔ 論文の対応関係と simulation 環境については `../README.md` を参照。